In [25]:
import numpy as np
import tifffile
import glob
import skimage.util as util
import cv2
import dask
import pandas as pd
import plotly.express as px
import skimage as ski
import os

In [26]:
fnames = glob.glob('*/*.tif')

In [27]:
def shrink_labels(label_image, size=3):
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))

    shrunken_label_image = np.zeros_like(label_image)
    regions = ski.measure.regionprops(label_image)

    for region in regions:
        lbl = region.label
        minr, minc, maxr, maxc = region.bbox

        minr = max(minr - 1, 0)
        minc = max(minc - 1, 0)
        maxr = min(maxr + 1, label_image.shape[0])
        maxc = min(maxc + 1, label_image.shape[1])

        # Extract the region from the original label image
        region_mask = label_image[minr:maxr, minc:maxc] == lbl

        # Erode the region
        eroded_region_mask = cv2.erode(region_mask.astype(np.uint8), kernel)

        # Update the shrunken_label_image
        shrunken_label_image[minr:maxr, minc:maxc][eroded_region_mask == 1] = lbl

    return shrunken_label_image

In [28]:
def backsub(inp, diameter=70):
    filterSize =(diameter, diameter)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                    filterSize)
    blurred = cv2.GaussianBlur(inp, (5, 5), 0)
    tophat_img = cv2.morphologyEx(blurred,
                                cv2.MORPH_TOPHAT,
                                kernel)
    rtn = inp.astype(np.single) - (blurred-tophat_img)
    rtn = np.clip(rtn, 0, np.inf)

    return rtn

In [29]:
def ptiles(regionmask, intensity):
    return np.percentile(intensity[regionmask], q=(50, 75, 90, 95, 99))
def std(regionmask, intensity):
    return np.std(intensity[regionmask])

In [30]:
def get_pearsons(channel0, channel1, labels):
    pearsons = []
    regions = ski.measure.regionprops(labels)
    labs = []

    for region in regions:
        lbl = region.label
        minr, minc, maxr, maxc = region.bbox
        if (maxr-minr<2) | (maxc-minc<2):
            continue

        labs.append(lbl)
        # Extract the region from the original label image
        region_mask = labels[minr:maxr, minc:maxc] == lbl

        corr, _ = ski.measure.pearson_corr_coeff(channel0[minr:maxr, minc:maxc], channel1[minr:maxr, minc:maxc], region_mask)
        pearsons.append(corr)
    return pd.DataFrame({'label': labs, 'pearson': pearsons})


In [31]:
def get_pearsons_filtered(channel0, channel1, labels):
    pearsons = []
    regions = ski.measure.regionprops(labels)
    labs = []

    for region in regions:
        lbl = region.label
        minr, minc, maxr, maxc = region.bbox
        if (maxr-minr<2) | (maxc-minc<2):
            continue

        labs.append(lbl)
        # Extract the region from the original label image
        region_mask = labels[minr:maxr, minc:maxc] == lbl
        c0_sub = channel0[minr:maxr, minc:maxc].copy()
        c1_sub = channel1[minr:maxr, minc:maxc].copy()
        percentile_threshold = 34
        c0_ptile = np.percentile(c0_sub[region_mask], percentile_threshold)
        c1_ptile = np.percentile(c1_sub[region_mask], percentile_threshold)
        c0_sub[c0_sub < c0_ptile] = 0
        c1_sub[c1_sub < c1_ptile] = 0

        corr, _ = ski.measure.pearson_corr_coeff(c0_sub, c1_sub, region_mask)
        pearsons.append(corr)
    return pd.DataFrame({'label': labs, 'pearson_filtered': pearsons})

In [32]:
def process_file(fname):
    # If File exists, skip
    #if os.path.exists(fname.replace('.tif', '.csv')):
    #    return
    import cellpose.models as models

    model = models.Cellpose(gpu=True, model_type='cyto')
    img = ski.io.imread(fname)
    img = np.moveaxis(img, -1, 0)

    img[0] = backsub(img[0])
    img[1] = backsub(img[1])

    # Do it on transmitted
    cells = model.eval(img[-1], diameter=60, channels=[0, 0])
    # Do it on mEos2
    #cells = model.eval(np.sqrt(img[1]), diameter=60, channels=[0, 0])
    
    cells_shrunk = shrink_labels(cells[0], size=12)

    props0 = pd.DataFrame(ski.measure.regionprops_table(cells_shrunk, img[0], properties=('label', 'area', 'mean_intensity', 'centroid')))
    props1 = pd.DataFrame(ski.measure.regionprops_table(cells_shrunk, img[1], properties=('label', 'mean_intensity')))
    df = pd.merge(props0, props1, on='label')

    # Get percentiles for channel 0
    props0 = ski.measure.regionprops(cells_shrunk, intensity_image=img[0], extra_properties=(ptiles,std))
    labels0 = np.array([p.label for p in props0])
    p50_0 = np.array([p.ptiles[0] for p in props0])
    p75_0 = np.array([p.ptiles[1] for p in props0])
    p90_0 = np.array([p.ptiles[2] for p in props0])
    p95_0 = np.array([p.ptiles[3] for p in props0])
    p99_0 = np.array([p.ptiles[4] for p in props0])
    stds_0 = np.array([p.std for p in props0])

    # Get percentiles for channel 1
    props1 = ski.measure.regionprops(cells_shrunk, intensity_image=img[1], extra_properties=(ptiles,std))
    labels1 = np.array([p.label for p in props1])
    p50_1 = np.array([p.ptiles[0] for p in props1])
    p75_1 = np.array([p.ptiles[1] for p in props1])
    p90_1 = np.array([p.ptiles[2] for p in props1])
    p95_1 = np.array([p.ptiles[3] for p in props1])
    p99_1 = np.array([p.ptiles[4] for p in props1])
    stds_1 = np.array([p.std for p in props1])

    tdf = pd.DataFrame({'label': labels0, 'p50_0': p50_0, 'p75_0': p75_0, 'p90_0': p90_0, 'p95_0': p95_0, 'p99_0': p99_0, 'std_0': stds_0, 'p50_1': p50_1, 'p75_1': p75_1, 'p90_1': p90_1, 'p95_1': p95_1, 'p99_1': p99_1, 'std_1': stds_1})
    df = pd.merge(df, tdf, on='label')

    # Get pearsons and pearsons filtered by top 80 pct brightest pixels
    pearsons_df = get_pearsons(img[0], img[1], cells_shrunk)
    df = pd.merge(df, pearsons_df, on='label')
    pearsons_filtered_df = get_pearsons_filtered(img[0], img[1], cells_shrunk)
    df = pd.merge(df, pearsons_filtered_df, on='label')
    df['file'] = fname
    df.to_csv(fname.replace('.tif', '.csv'))
    ski.io.imsave(fname+'ff', cells_shrunk)
    
    return df

In [33]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import dask
import time
cluster = SLURMCluster(
    cores=1,                      # Number of cores per job
    memory="16GB",                # Memory per job
    queue="compute",         # Queue/partition name
    walltime="24:00:00",           # Job time limit
    local_directory="$TMPDIR",     # Temporary directory (optional)
    log_directory="logs",          # Directory for log files (optional)
)
client = Client(cluster)
print(client)
cluster.scale(jobs=512)  # Request 4 jobs

<Client: 'tcp://10.0.49.128:42975' processes=0 threads=0, memory=0 B>


In [34]:
# from dask_jobqueue import SLURMCluster
# from dask.distributed import Client
# import dask
# import time
# cluster = SLURMCluster(
#     cores=1,                      # Number of cores per job
#     memory="64GB",                # Memory per job
#     queue="gpu",         # Queue/partition name
#      job_extra_directives =[
#         '--gres=gpu:a100:1',  # Number of GPUs per job
#     ],
#     walltime="24:00:00",           # Job time limit
#     local_directory="$TMPDIR",     # Temporary directory (optional)
#     log_directory="logs",          # Directory for log files (optional)
# )
# client = Client(cluster)
# print(client)
# cluster.scale(jobs=4)  # Request 4 jobs

In [35]:
all_data = []
delayed = dask.delayed(process_file)
for i, f in enumerate(fnames):
    all_data.append(delayed(f))
all_data = dask.compute(*all_data)

In [36]:
client.close()
cluster.close()